# 01_build_canonical_pipeline

Notebook conversion of `01_build_canonical_pipeline.py`.

- Original computational/output logic is preserved.
- `__file__`-dependent paths are adapted for Jupyter execution.
- Requires `Arab_Light_Kinetics_Source_FWO_KAS_Starink.xlsx` and `Arab_Light_TGA_Canonical_Data.xlsx` in `data/private/` (withheld from this pre-acceptance release; see `data/README.md`).


In [1]:
"""01_build_canonical_pipeline.py — implements the documented canonical
pipeline (workbook README + ML manuscript Methods + Supplementary Table S1)
and validates every T_alpha against the frozen Kinetic_Talpha sheet.

Documented steps (three independent, mutually-consistent sources):
  1. Time order preserved.
  2. Temperature: cumulative maximum (not sorting) removes backward steps.
  3. Interpolate onto a UNIFORM 1 K grid, 323-1000 K (full domain, no
     ramp-only trimming -- this is the key correction vs. prior sessions).
  4. Air: terminal baseline correction only (buoyancy/drift), preserving
     low-temperature oxygen-uptake mass gain. N2: no baseline correction
     needed (raw terminal offset ~0).
  5. Savitzky-Golay smoothing, window=21, polyorder=3 (FIXED, matches
     Table S2/Methods -- not the 15/21/31 sweep from the earlier,
     incorrectly-scoped sensitivity check).
  6. N2: isotonic regression enforces non-increasing mass. Air: NOT
     constrained (per Supplementary S1, verbatim) -- no artifact repair
     of any kind applied here, by design.
  7. DTG = -d(mass)/dT on the clean grid.
  8. alpha = (m0 - mt) / (m0 - mf), with m0 = 100.0% from the true
     initial instrument reading and mf = the cleaned mass at T=1000 K.

Acceptance test: every computed T_alpha must match the frozen
Kinetic_Talpha sheet to within a small numerical tolerance.
"""

from __future__ import annotations

import sys
from pathlib import Path

# Notebook-safe replacement for Path(__file__).resolve().parent
CODE_DIR = Path.cwd().resolve()
if not (CODE_DIR / "alc_core.py").exists() and (CODE_DIR / "03_CODE" / "alc_core.py").exists():
    CODE_DIR = CODE_DIR / "03_CODE"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import numpy as np
import pandas as pd
import openpyxl
from alc_core import BETAS, RESULTS_DIR, DATA_DIR, GRID, canonical_clean_curve, first_crossing_T

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SG_WINDOW = 21
SG_POLYORDER = 3
ALPHA_TARGETS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]


# ---- run pipeline for all 6 curves ----
results = {}
disclosure_rows = []
for atm in ("N2", "Air"):
    for beta in BETAS:
        curve, terminal_offset = canonical_clean_curve(atm, beta, sg_window=SG_WINDOW, baseline_mode="constant_1000K")
        m0 = 100.0  # true raw initial mass (t=0 reading) -- confirmed correct
        mf = curve.mass_clean_pct.iloc[-1]  # resulting cleaned final plateau,
                    # not the manually forced Table S1 value -- now that the
                    # baseline anchor bug is fixed, the pipeline's own
                    # resulting plateau is the right reference (see delta
                    # vs Table S1 residues printed below for comparison)
        alpha = (m0 - curve.mass_clean_pct.to_numpy()) / (m0 - mf)  # step 8
        results[(atm, beta)] = {"curve": curve, "m0": m0, "mf": mf, "alpha": alpha}
        disclosure_rows.append({
            "atmosphere": atm, "beta": beta, "m0_cleaned_pct": m0, "mf_cleaned_pct": mf,
            "raw_terminal_offset_pct": terminal_offset,
        })

print("=== m0 / mf per curve (step 8 reference points) ===")
print(pd.DataFrame(disclosure_rows).to_string(index=False))

# ---- extract T_alpha for every target ----
talpha_rows = []
for atm in ("N2", "Air"):
    for a in ALPHA_TARGETS:
        row = {"atmosphere": atm, "alpha": a}
        for beta in BETAS:
            r = results[(atm, beta)]
            T_a = first_crossing_T(GRID, r["alpha"], a)
            row[f"T_beta{beta}"] = T_a
        talpha_rows.append(row)
computed = pd.DataFrame(talpha_rows)

# ---- load canonical Kinetic_Talpha sheet for comparison ----
wb = openpyxl.load_workbook(DATA_DIR / "Arab_Light_TGA_Canonical_Data.xlsx", data_only=True)
ws = wb["Kinetic_Talpha"]
canon_rows = []
for row in ws.iter_rows(min_row=4, max_row=19, values_only=True):
    if row[0] is None:
        continue
    canon_rows.append({"atmosphere": row[0], "alpha": row[1],
                       "T_beta5_canon": row[2], "T_beta10_canon": row[3], "T_beta20_canon": row[4]})
canon = pd.DataFrame(canon_rows)

check = computed.merge(canon, on=["atmosphere", "alpha"])
for beta in BETAS:
    check[f"delta_beta{beta}"] = check[f"T_beta{beta}"] - check[f"T_beta{beta}_canon"]

print("\n=== VALIDATION: computed T_alpha vs. frozen Kinetic_Talpha sheet ===")
cols = ["atmosphere", "alpha"] + [f"T_beta{b}" for b in BETAS] + [f"T_beta{b}_canon" for b in BETAS] + [f"delta_beta{b}" for b in BETAS]
print(check[cols].to_string(index=False))

max_delta = check[[f"delta_beta{b}" for b in BETAS]].abs().max().max()
main_region = check[check.alpha >= 0.2]  # alpha=0.1 is documented as ill-conditioned /
                                          # excluded from fits in both the workbook README
                                          # and Supplementary text -- large deltas there
                                          # are expected and not part of the acceptance test
max_delta_main = main_region[[f"delta_beta{b}" for b in BETAS]].abs().max().max()
print(f"\nMax |delta T_alpha| across ALL rows (incl. excluded alpha=0.1): {max_delta:.4f} K")
print(f"Max |delta T_alpha| across alpha=0.2-0.8 (the actual fitting region): {max_delta_main:.4f} K")
tolerance = 0.5  # K
print(f"Main-region result: {'PASS' if max_delta_main <= tolerance else 'FAIL'} (within {tolerance} K)")

check.to_csv(RESULTS_DIR / "Canonical_Talpha_validation.csv", index=False)
pd.DataFrame(disclosure_rows).to_csv(RESULTS_DIR / "Canonical_pipeline_m0_mf.csv", index=False)
print(f"\nWrote Canonical_Talpha_validation.csv and Canonical_pipeline_m0_mf.csv")


=== m0 / mf per curve (step 8 reference points) ===
atmosphere  beta  m0_cleaned_pct  mf_cleaned_pct  raw_terminal_offset_pct
        N2     5           100.0        2.628718                 0.000000
        N2    10           100.0        2.666481                 0.000000
        N2    20           100.0        3.102807                 0.000000
       Air     5           100.0       -0.000275                -0.045640
       Air    10           100.0       -0.000072                -0.316204
       Air    20           100.0       -0.000302                 0.319437



=== VALIDATION: computed T_alpha vs. frozen Kinetic_Talpha sheet ===
atmosphere  alpha    T_beta5   T_beta10   T_beta20  T_beta5_canon  T_beta10_canon  T_beta20_canon  delta_beta5  delta_beta10  delta_beta20
        N2    0.1 327.808169 327.776564 327.835607        327.808         327.776         327.830     0.000169      0.000564      0.005607
        N2    0.2 406.031276 426.296337 445.528161        406.028         426.291         445.526     0.003276      0.005337      0.002161
        N2    0.3 447.291683 469.212802 489.165839        447.288         469.206         489.164     0.003683      0.006802      0.001839
        N2    0.4 485.187141 508.003354 527.929455        485.182         507.994         527.927     0.005141      0.009354      0.002455
        N2    0.5 523.458720 546.975197 566.729810        523.453         546.963         566.726     0.005720      0.012197      0.003810
        N2    0.6 563.760595 587.816973 607.393355        563.753         587.802         607.38